In [1]:
import yaml
import numpy as np
from scipy.spatial.transform import Rotation as R

def rotmat_from_quat(q):
    q = np.array(q)
    q = q/np.linalg.norm(q)
    return R.from_quat(q).as_matrix()

def calc_actuator_mixing_matrix(config_file):
    with open(config_file, 'r') as stream:
        try:
            config = yaml.safe_load(stream)
        except yaml.YAMLError as exc:
            print(exc)

    n_motors = len(config)
    actuator_mixing_matrix = np.zeros((6,n_motors))
    for i in range(n_motors):
        vec_z_mf = np.array([0,0,1]) # motor frame z axis
        q_i = np.array(config['motor_'+str(i)]['orientation'])
        # rotation matrix from quaternion [x,y,z,w]
        rotmat_i = rotmat_from_quat(q_i)
        t_i = np.array(config['motor_'+str(i)]['translation'])
        torque_direction = -config['motor_'+str(i)]['direction']
        torque_constant = config['motor_'+str(i)]['torque_constant']
        vec_z_bf = rotmat_i @ vec_z_mf # motor axis in body frame
        # forces
        actuator_mixing_matrix[:3,i] = vec_z_bf
        # torques
        actuator_mixing_matrix[3:,i] = np.cross(t_i, vec_z_bf) + vec_z_bf*torque_constant*torque_direction
    return actuator_mixing_matrix

In [2]:
filename = 'motor_config.yaml'
mixing_matrix = calc_actuator_mixing_matrix(filename)
print(mixing_matrix)


[[ 0.      0.      0.      0.      0.      0.    ]
 [ 0.      0.      0.      0.      0.      0.    ]
 [ 1.      1.      1.      1.      1.      1.    ]
 [-0.0785 -0.0785  0.0785  0.0785 -0.0785  0.0785]
 [-0.0785  0.0785  0.0785 -0.0785  0.      0.    ]
 [-0.01   -0.01    0.01    0.01    0.01   -0.01  ]]


['motor_0', 'motor_1', 'motor_2', 'motor_3', 'motor_4', 'motor_5']


In [3]:
import urdfpy
import torch

filename = "/home/vimarsh/workspaces/aerial_gym_ws/src/aerial_gym_simulator/resources/robots/hexa/hexa2.urdf"
urdf = urdfpy.URDF.load(filename)
print(urdf)
n_motors = 6
motor_names = ['motor_'+str(i) for i in range(n_motors)]
print(motor_names)

# get transformation of motor link wrt base link
motor_transforms = {}
links_fk = urdf.link_fk()
for key, value in links_fk.items():
    if key.name in motor_names:
        transform = value
        # print(key.name, transform)
        print(key.name, transform[:, 3])
        r, p, y = urdfpy.matrix_to_rpy(transform)
        print(key.name, r, p, y)

motor_poses = []
motor_torque_constants = []
motor_directions = []

for motor_name in motor_names:
    for link, transform in links_fk.items():
        if link.name == motor_name:
            motor_poses.append(transform[:3, :].flatten())
            motor_torque_constants.append(0.01)  # Assuming torque constant is same for all motors
            motor_directions.append(-1 if motor_names.index(motor_name) < 4 and motor_names.index(motor_name) % 2 == 0 else 1)

motor_poses = torch.tensor(motor_poses, dtype=torch.float32).cuda()
motor_torque_constants = torch.tensor(motor_torque_constants, dtype=torch.float32).cuda()
motor_directions = torch.tensor(motor_directions, dtype=torch.float32).cuda()

print(motor_poses)
print(motor_torque_constants)
print(motor_directions)


['motor_0', 'motor_1', 'motor_2', 'motor_3', 'motor_4', 'motor_5']
motor_0 [ 0.13 -0.13  0.    1.  ]
motor_0 0.0 -0.0 0.0
motor_1 [-0.13 -0.13  0.    1.  ]
motor_1 0.0 -0.0 0.0
motor_2 [-0.13  0.13  0.    1.  ]
motor_2 0.0 -0.0 0.0
motor_3 [0.13 0.13 0.   1.  ]
motor_3 0.0 -0.0 0.0
motor_4 [0.   0.17 0.   1.  ]
motor_4 0.0 -0.0 0.0
motor_5 [ 0.   -0.17  0.    1.  ]
motor_5 0.0 -0.0 0.0


/tmp/ipykernel_5697/2801827721.py:33: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at ../torch/csrc/utils/tensor_new.cpp:278.)
  motor_poses = torch.tensor(motor_poses, dtype=torch.float32).cuda()


tensor([[ 1.0000,  0.0000,  0.0000,  0.1300,  0.0000,  1.0000,  0.0000, -0.1300,
          0.0000,  0.0000,  1.0000,  0.0000],
        [ 1.0000,  0.0000,  0.0000, -0.1300,  0.0000,  1.0000,  0.0000, -0.1300,
          0.0000,  0.0000,  1.0000,  0.0000],
        [ 1.0000,  0.0000,  0.0000, -0.1300,  0.0000,  1.0000,  0.0000,  0.1300,
          0.0000,  0.0000,  1.0000,  0.0000],
        [ 1.0000,  0.0000,  0.0000,  0.1300,  0.0000,  1.0000,  0.0000,  0.1300,
          0.0000,  0.0000,  1.0000,  0.0000],
        [ 1.0000,  0.0000,  0.0000,  0.0000,  0.0000,  1.0000,  0.0000,  0.1700,
          0.0000,  0.0000,  1.0000,  0.0000],
        [ 1.0000,  0.0000,  0.0000,  0.0000,  0.0000,  1.0000,  0.0000, -0.1700,
          0.0000,  0.0000,  1.0000,  0.0000]], device='cuda:0')
tensor([0.0100, 0.0100, 0.0100, 0.0100, 0.0100, 0.0100], device='cuda:0')
tensor([-1.,  1., -1.,  1.,  1.,  1.], device='cuda:0')


In [12]:
# Calculate mixing matrix from URDF transforms
n_motors = 6
actuator_mixing_matrix = np.zeros((6, n_motors))
torque_constant = 0.01  #same for all 
vec_z_mf = np.array([0, 0, 1])  # motor frame z axis

for i in range(n_motors):
    motor_name = f'motor_{i}'
    for link, transform in links_fk.items():
        if link.name == motor_name:
            rotation_matrix = transform[:3, :3]
            translation = transform[:3, 3]

            # print(f"Motor {i} transform:")
            # print(rotation_matrix)
            # print(translation)
            
            # alternating directions between motors (for first 4), for other two just same direction 
            torque_direction = -1 if i < 4 and i % 2 == 0 else 1
            # torque_direction = -1 if i % 2 == 0 else 1
            
            #thrust vector in body frame
            vec_z_bf = rotation_matrix @ vec_z_mf
            
            #forces (first 3 rows)
            actuator_mixing_matrix[:3, i] = vec_z_bf
            
            #toeque (last 3 rows)
            actuator_mixing_matrix[3:, i] = np.cross(translation, vec_z_bf) + vec_z_bf * torque_constant * torque_direction

print(actuator_mixing_matrix)

[[ 0.    0.    0.    0.    0.    0.  ]
 [ 0.    0.    0.    0.    0.    0.  ]
 [ 1.    1.    1.    1.    1.    1.  ]
 [-0.13 -0.13  0.13  0.13  0.17 -0.17]
 [-0.13  0.13  0.13 -0.13  0.    0.  ]
 [-0.01  0.01 -0.01  0.01  0.01  0.01]]


In [15]:
import torch
import numpy as np
from scipy.spatial.transform import Rotation as R

def quat_rotate_multidim(quaternions: torch.Tensor, vectors: torch.Tensor) -> torch.Tensor:
    """Rotate 3D vectors using batched quaternions with multi-dimensional support.
    
    Args:
        quaternions: Tensor of shape (..., 4) - must match vector dimensions except last
        vectors: Tensor of shape (..., 3) - must match quaternion dimensions except last
    
    Returns:
        Rotated vectors tensor of same shape as input vectors
    """
    # Reshape for batch processing while preserving robot-motor structure
    original_shape = vectors.shape
    quats = quaternions.reshape(-1, 4)  # Flatten all leading dimensions
    vecs = vectors.reshape(-1, 3)      # Flatten all leading dimensions
    
    # Normalize quaternions for stable rotation
    quats = quats / torch.norm(quats, dim=-1, keepdim=True)

    # Convert vectors to quaternion form (0 real part)
    vec_quats = torch.cat([torch.zeros_like(vecs[..., :1]), vecs], dim=-1)
    
    # Compute rotation: q * v * q_conj
    q_conj = torch.cat([quats[..., :1], -quats[..., 1:]], dim=-1)
    rotated = torch.stack([
        quats[:, 0] * vec_quats[:, 0] - quats[:, 1] * vec_quats[:, 1] - quats[:, 2] * vec_quats[:, 2] - quats[:, 3] * vec_quats[:, 3],
        quats[:, 0] * vec_quats[:, 1] + quats[:, 1] * vec_quats[:, 0] + quats[:, 2] * vec_quats[:, 3] - quats[:, 3] * vec_quats[:, 2],
        quats[:, 0] * vec_quats[:, 2] - quats[:, 1] * vec_quats[:, 3] + quats[:, 2] * vec_quats[:, 0] + quats[:, 3] * vec_quats[:, 1],
        quats[:, 0] * vec_quats[:, 3] + quats[:, 1] * vec_quats[:, 2] - quats[:, 2] * vec_quats[:, 1] + quats[:, 3] * vec_quats[:, 0]
    ], dim=-1)[..., 1:]  # Extract vector part

    return rotated.reshape(original_shape)



def calculate_allocation_matrix(motor_poses, motor_torque_constants, motor_directions):
    num_robots = motor_poses.shape[0]
    num_motors = motor_poses.shape[1]
    allocation_matrix = torch.zeros((num_robots, 6, num_motors), dtype=torch.float32).cuda()

    motor_force_frame = torch.tensor([0.0, 0.0, 1.0], dtype=torch.float32).cuda().unsqueeze(0).expand(num_robots, num_motors, -1)
    # print(motor_force_frame.shape)

    motor_torque_constants = motor_torque_constants.unsqueeze(-1)
    # print(motor_torque_constants.shape)

    motor_torque = motor_force_frame * motor_torque_constants.unsqueeze(-1)

    # print(motor_poses)
    motor_pos = motor_poses[..., 0:3]
    motor_quat = motor_poses[..., 3:7]

    # print(motor_quat.shape)
    # print(motor_force_frame.shape, motor_quat.shape)
    # print(motor_torque.shape, motor_directions.shape)
    # Transform force and torque to robot frame
    motor_force_in_robot_frame = quat_rotate_multidim(motor_quat, motor_force_frame)
    motor_torque_in_robot_frame = quat_rotate_multidim(motor_quat, motor_torque*(-motor_directions.unsqueeze(-1)))

    # Calculate torque as r x F + tau
    torque = torch.cross(motor_pos, motor_force_in_robot_frame) + motor_torque_in_robot_frame

    # print(motor_force_in_robot_frame.shape, torque.shape)
    # Fill the allocation matrix
    print(allocation_matrix.shape)
    allocation_matrix[:, 0:3, :] = motor_force_in_robot_frame.transpose(1, 2)
    allocation_matrix[:, 3:6, :] = torque.transpose(1, 2)
    return allocation_matrix



In [20]:
allocation_matrix = calculate_allocation_matrix(motor_poses, motor_torque_constants, motor_directions)
print(allocation_matrix)

RuntimeError: The size of tensor a (6) must match the size of tensor b (72) at non-singleton dimension 0

In [19]:
def test_quat_rotation():
    # Test single robot with 12 motors
    q = torch.rand(6, 12, 4).cuda()
    v = torch.rand(6, 12, 3).cuda()
    
    # Should return same shape as input vectors
    rotated = quat_rotate_multidim(q, v)
    assert rotated.shape == v.shape
    
    # Test identity rotation
    identity_quat = torch.tensor([1.0, 0.0, 0.0, 0.0], device='cuda').repeat(6, 12, 1)
    rotated = quat_rotate_multidim(identity_quat, v)
    assert torch.allclose(rotated, v, atol=1e-6)
    print("Quaternion rotation test passed!")

test_quat_rotation()

Quaternion rotation test passed!
